# Propofol Control RL Environment Validation

This notebook runs synthetic, training-free validation of the Gymnasium environment.

> This environment is a research-only reconstruction around a published PK-PD simulator. It is not a medical device and must not be used for clinical dosing or patient care.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path('/content/VitalDB-Feature-Selection')
if not repo.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/khyitty/VitalDB-Feature-Selection.git',
        str(repo),
    ], check=True)
os.chdir(repo)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gymnasium', 'numpy', 'pandas', 'scipy', 'matplotlib',
], check=True)
print('Repository:', repo)
print('Commit:', subprocess.run(
    ['git', 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True
).stdout.strip())

In [ ]:
from src.rl_env.state_adapters import STATE_PROFILES

for name, profile in STATE_PROFILES.items():
    print(f'{name}: history=(6, {len(profile.dynamic_feature_names)}), static=(4,)')
    print('  ', profile.dynamic_feature_names)

In [ ]:
output_dir = repo / 'outputs' / 'rl_environment_validation'
command = [
    sys.executable, 'scripts/validate_rl_environment.py',
    '--state-profile', 'attention_ready',
    '--patient-profile', 'middle_male',
    '--target-bis', '50',
    '--episode-duration-seconds', '600',
    '--action-schedule', 'step',
    '--remifentanil-schedule', 'piecewise',
    '--action-bounds-profile', 'synthetic_nonclinical_v1',
    '--reward-profile', 'transparent_tracking_v1',
    '--seed', '20260716',
    '--output-dir', str(output_dir),
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = json.loads((output_dir / 'validation_summary.json').read_text())
metrics = json.loads((output_dir / 'episode_metrics.json').read_text())
equivalence = pd.read_csv(output_dir / 'profile_equivalence.csv')
print(json.dumps(summary, indent=2))
print(json.dumps(metrics, indent=2))
display(equivalence)

In [ ]:
from IPython.display import Image, display

for figure in sorted((output_dir / 'figures').glob('*.png')):
    print(figure.name)
    display(Image(filename=str(figure)))

## Boundary

The actions above are fixed scripted schedules, not learned policies. No policy optimization or real-patient evaluation is performed.